# Create Keras DNN fraud model.

### Load configuration settings from the setup notebook

Set the constants used in this notebook and load the config settings from the `00_environment_setup.ipynb` notebook.

In [4]:
GCP_PROJECTS = !gcloud config get-value project
PROJECT_ID = GCP_PROJECTS[0]
BUCKET_NAME = f"{PROJECT_ID}-fraudfinder"
config = !gsutil cat gs://{BUCKET_NAME}/config/notebook_env.py
print(config.n)
exec(config.n)


BUCKET_NAME          = "qwiklabs-asl-04-fb8d9ff3cdaf-fraudfinder"
PROJECT              = "qwiklabs-asl-04-fb8d9ff3cdaf"
REGION               = "us-central1"
ID                   = "ozrwj"
FEATURESTORE_ID      = "fraudfinder_ozrwj"
MODEL_NAME           = "ff_model"
ENDPOINT_NAME        = "ff_model_endpoint"
TRAINING_DS_SIZE     = "1000"



### Import libraries and define constants

#### Libraries
Here, we'll import the necessary libraries for this notebook.

In [5]:
# General
import datetime as dt
import json
import os
import random
import sys
import time
from datetime import datetime, timedelta
from typing import List, Union

# Data Engineering
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 500)

# Vertex AI and Vertex AI Feature Store
from google.cloud import aiplatform as vertex_ai
from google.cloud import bigquery

/home/jupyter/asl-ml-immersion/asl_mlops/.venv/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


### Helpers

Define a set of helper functions to run BigQuery query and create features. 

In [ ]:
def run_bq_query(sql: str, show=False) -> Union[str, pd.DataFrame]:
    """
    Run a BigQuery query and return the job ID or result as a DataFrame
    Args:
        sql: SQL query, as a string, to execute in BigQuery
        show: A flag to show query result in a Pandas Dataframe
    Returns:
        df: DataFrame of results from query,  or error, if any
    """

    bq_client = bigquery.Client()

    # Try dry run before executing query to catch any errors
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    bq_client.query(sql, job_config=job_config)

    # If dry run succeeds without errors, proceed to run query
    job_config = bigquery.QueryJobConfig()
    client_result = bq_client.query(sql, job_config=job_config)

    job_id = client_result.job_id

    # Wait for query/job to finish running. then get & return data frame
    result = client_result.result()
    print(f"Finished job_id: {job_id}")

    if show:
        df = result.to_arrow().to_pandas()
        return df

### Export train dataset

In [ ]:
export_train_dataset = f"""
EXPORT DATA OPTIONS(
      uri='gs://{BUCKET_NAME}/data_export/manual_export_train/*.csv',
      format='csv',
      header=true
    ) AS
    SELECT 
    transaction_id,
    tx_fraud,
    tx_amount,
    customer_id_nb_tx_14day_window,
    customer_id_nb_tx_7day_window,
    customer_id_nb_tx_1day_window,
    customer_id_avg_amount_14day_window,
    customer_id_avg_amount_7day_window,
    customer_id_avg_amount_1day_window,
    terminal_id_nb_tx_14day_window,
    terminal_id_nb_tx_7day_window,
    terminal_id_nb_tx_1day_window,
    terminal_id_risk_14day_window,
    terminal_id_risk_7day_window,
    terminal_id_risk_1day_window,
    customer_id_nb_tx_60min_window,
    customer_id_nb_tx_30min_window,
    customer_id_nb_tx_15min_window,
    customer_id_avg_amount_60min_window,
    customer_id_avg_amount_30min_window,
    customer_id_avg_amount_15min_window,
    terminal_id_nb_tx_60min_window,
    terminal_id_nb_tx_30min_window,
    terminal_id_nb_tx_15min_window,
    terminal_id_avg_amount_60min_window,
    terminal_id_avg_amount_30min_window,
    terminal_id_avg_amount_15min_window
    FROM `tx.ff_join_tx_features`
    WHERE 
    train_split='train'
    AND (tx_fraud = 1 OR (tx_fraud = 0 AND MOD(ABS(FARM_FINGERPRINT(transaction_id)), 100) < 2))
"""
print(export_train_dataset)

In [ ]:
run_bq_query(export_train_dataset)

### Export validation dataset

In [ ]:
export_valid_dataset = f"""
EXPORT DATA OPTIONS(
      uri='gs://{BUCKET_NAME}/data_export/manual_export_valid/*.csv',
      format='csv',
      header=true
    ) AS
    SELECT 
    transaction_id,
    tx_fraud,
    tx_amount,
    customer_id_nb_tx_14day_window,
    customer_id_nb_tx_7day_window,
    customer_id_nb_tx_1day_window,
    customer_id_avg_amount_14day_window,
    customer_id_avg_amount_7day_window,
    customer_id_avg_amount_1day_window,
    terminal_id_nb_tx_14day_window,
    terminal_id_nb_tx_7day_window,
    terminal_id_nb_tx_1day_window,
    terminal_id_risk_14day_window,
    terminal_id_risk_7day_window,
    terminal_id_risk_1day_window,
    customer_id_nb_tx_60min_window,
    customer_id_nb_tx_30min_window,
    customer_id_nb_tx_15min_window,
    customer_id_avg_amount_60min_window,
    customer_id_avg_amount_30min_window,
    customer_id_avg_amount_15min_window,
    terminal_id_nb_tx_60min_window,
    terminal_id_nb_tx_30min_window,
    terminal_id_nb_tx_15min_window,
    terminal_id_avg_amount_60min_window,
    terminal_id_avg_amount_30min_window,
    terminal_id_avg_amount_15min_window
    FROM `tx.ff_join_tx_features`
    WHERE 
        train_split='validation'
    AND MOD(ABS(FARM_FINGERPRINT(transaction_id)), 100) < 10
"""
print(export_valid_dataset)

In [ ]:
run_bq_query(export_valid_dataset)

## Create Keras model

In [ ]:
def train_model(
    project_id: str,          
    model_output_uri: str,
    train_files_pattern: str,
    validation_files_pattern: str
):
    CSV_COLUMNS = [
        'transaction_id', 'tx_fraud', 
        'tx_amount',
        'customer_id_nb_tx_14day_window', 'customer_id_nb_tx_7day_window', 'customer_id_nb_tx_1day_window',
        'customer_id_avg_amount_14day_window', 'customer_id_avg_amount_7day_window', 'customer_id_avg_amount_1day_window',
        'terminal_id_nb_tx_14day_window', 'terminal_id_nb_tx_7day_window', 'terminal_id_nb_tx_1day_window',
        'terminal_id_risk_14day_window', 'terminal_id_risk_7day_window', 'terminal_id_risk_1day_window',
        'customer_id_nb_tx_60min_window', 'customer_id_nb_tx_30min_window', 'customer_id_nb_tx_15min_window',
        'customer_id_avg_amount_60min_window', 'customer_id_avg_amount_30min_window', 'customer_id_avg_amount_15min_window',
        'terminal_id_nb_tx_60min_window', 'terminal_id_nb_tx_30min_window', 'terminal_id_nb_tx_15min_window',
        'terminal_id_avg_amount_60min_window', 'terminal_id_avg_amount_30min_window', 'terminal_id_avg_amount_15min_window'
    ]

    import os
    os.environ["KERAS_BACKEND"] = "tensorflow"
    import keras
    import numpy as np
    import tensorflow as tf
    from sklearn.metrics import roc_curve, precision_recall_curve, auc, confusion_matrix, matthews_corrcoef, f1_score


    target_column_name='tx_fraud'
    features_columns_offest = 2
    input_feature_names = CSV_COLUMNS[features_columns_offest:] 
    RECORD_DEFAULTS = [[''], [0]] + [[0.0]] * (len(CSV_COLUMNS) - features_columns_offest)
    
    def parse_csv_row(csv_row):
        columns = tf.io.decode_csv(csv_row, record_defaults=RECORD_DEFAULTS)
        features = dict(zip(CSV_COLUMNS, columns))

        #remove transaction_id from features
        features.pop('transaction_id')

        #remove target from features
        label = tf.expand_dims(features.pop(target_column_name), axis=-1)
        
        for key in features.keys():
            features[key] = tf.expand_dims(features[key], axis=-1)
        return features, label

    batch_size = 256
    epochs = 10
    learning_rate=0.001
    
    train_files = tf.data.Dataset.list_files(train_files_pattern)
    train_dataset = train_files.interleave(
        lambda filename: tf.data.TextLineDataset(filename).skip(1),
        cycle_length=tf.data.AUTOTUNE,
        num_parallel_calls=tf.data.AUTOTUNE
    ).shuffle(buffer_size=1000).batch(batch_size)\
     .map(parse_csv_row, num_parallel_calls=tf.data.AUTOTUNE)\
     .repeat(epochs).prefetch(tf.data.AUTOTUNE)

    validation_files = tf.data.Dataset.list_files(validation_files_pattern)
    validation_dataset = validation_files.interleave(
        lambda filename: tf.data.TextLineDataset(filename).skip(1),
        cycle_length=tf.data.AUTOTUNE,
        num_parallel_calls=tf.data.AUTOTUNE
    ).batch(batch_size).map(parse_csv_row, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    inputs = [keras.Input(shape=(1,), name=name) for name in input_feature_names]
    x = keras.layers.concatenate(inputs)
    x = keras.layers.Dense(32, activation="relu")(x)
    x = keras.layers.Dense(16, activation="relu")(x)
    x = keras.layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs=inputs, outputs=x)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(), 
                 keras.metrics.AUC(curve="ROC", name="auc"), 
                 keras.metrics.AUC(curve="PR", name="pr")]
    )

    model.fit(
        train_dataset, 
        validation_data=validation_dataset, 
        epochs=epochs, 
        verbose=1
    )
    
    # Model serving input-ouput wrapper
    signature_dict = {col: tf.TensorSpec(shape=(None,), dtype=tf.float32, name=col) for col in input_feature_names}
    @tf.function(input_signature=[signature_dict])
    def serving_fn(json_inputs):
        prob_1 = model({col: tf.expand_dims(json_inputs[col], axis=1) for col in input_feature_names})
        prob_0 = 1.0 - prob_1
        return {"scores": tf.concat([prob_0, prob_1], axis=1), "classes": tf.tile(tf.constant([['0', '1']]), [tf.shape(prob_1)[0], 1])}
    
    export_archive = keras.export.ExportArchive()
    export_archive.track(model)
    export_archive.add_endpoint(name="serving_default", fn=serving_fn)
    export_archive.write_out(model_output_uri)

In [ ]:
train_files_pattern=f'gs://{BUCKET_NAME}/data_export/manual_export_train/*.csv'
valid_files_pattern=f'gs://{BUCKET_NAME}/data_export/manual_export_valid/*.csv'
model_output_uri=f'gs://{BUCKET_NAME}/manual_model_export/model_v1'

In [ ]:
train_model(
    PROJECT_ID,
    model_output_uri,
    train_files_pattern,
    valid_files_pattern
)

## Upload model, create endpoint and deploy trained model

Uploading our SavedModel from the above `MODEL_LOCATION`, creating and endpoint and deploying the trained model to act as a REST web service are three simple gcloud calls. We also run a command to list the endpoints, to fetch the fully qualified resource name `ENDPOINT_RESOURCENAME` for the endpoint.

In [6]:
os.environ["PROJECT"] = PROJECT_ID
os.environ["BUCKET_NAME"] = BUCKET_NAME
os.environ["REGION"] = REGION
os.environ["ENDPOINT_DISPLAYNAME"] = ENDPOINT_NAME
os.environ["MODEL_DISPLAYNAME"] = MODEL_NAME

In [ ]:
%%bash
TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
IMAGE_URI="us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"
MODEL_LOCATION=gs://${BUCKET_NAME}/manual_model_export/model_v1/
echo "MODEL_LOCATION=${MODEL_LOCATION}"
echo "MODEL_LOCATION=${MODEL_LOCATION}"

MODEL_RESOURCENAME=$(gcloud ai models upload \
    --region=$REGION \
    --display-name=$MODEL_DISPLAYNAME \
    --container-image-uri=$IMAGE_URI \
    --artifact-uri=$MODEL_LOCATION \
    --format="value(model)")

MODEL_ID=$(echo $MODEL_RESOURCENAME | cut -d"/" -f6)

echo "MODEL_DISPLAYNAME=${MODEL_DISPLAYNAME}"
echo "MODEL_RESOURCENAME=${MODEL_RESOURCENAME}"
echo "MODEL_ID=${MODEL_ID}"

# Endpoint
ENDPOINT_RESOURCENAME=$(gcloud ai endpoints create \
  --region=$REGION \
  --display-name=$ENDPOINT_DISPLAYNAME \
  --format="value(name)")

ENDPOINT_ID=$(echo $ENDPOINT_RESOURCENAME | cut -d"/" -f6)

echo "ENDPOINT_DISPLAYNAME=${ENDPOINT_DISPLAYNAME}"
echo "ENDPOINT_RESOURCENAME=${ENDPOINT_RESOURCENAME}"
echo "ENDPOINT_ID=${ENDPOINT_ID}"

# Deployment
DEPLOYEDMODEL_DISPLAYNAME=${MODEL_DISPLAYNAME}_deployment
MACHINE_TYPE=e2-standard-4
MIN_REPLICA_COUNT=1
MAX_REPLICA_COUNT=1

gcloud ai endpoints deploy-model $ENDPOINT_RESOURCENAME \
  --region=$REGION \
  --model=$MODEL_RESOURCENAME \
  --display-name=$DEPLOYEDMODEL_DISPLAYNAME \
  --machine-type=$MACHINE_TYPE \
  --min-replica-count=$MIN_REPLICA_COUNT \
  --max-replica-count=$MAX_REPLICA_COUNT \
  --traffic-split=0=100

MODEL_LOCATION=gs://qwiklabs-asl-04-fb8d9ff3cdaf-fraudfinder/manual_model_export/model_v1/
MODEL_LOCATION=gs://qwiklabs-asl-04-fb8d9ff3cdaf-fraudfinder/manual_model_export/model_v1/


Using endpoint [https://us-central1-aiplatform.googleapis.com/]
Waiting for operation [5589332682233348096]...
.....................................done.


MODEL_DISPLAYNAME=ff_model
MODEL_RESOURCENAME=projects/992044206639/locations/us-central1/models/4425103842996125696
MODEL_ID=4425103842996125696


Using endpoint [https://us-central1-aiplatform.googleapis.com/]
Waiting for operation [3790989056028966912]...
.....................done.
Created Vertex AI endpoint: projects/992044206639/locations/us-central1/endpoints/1416074769308057600.


ENDPOINT_DISPLAYNAME=ff_model_endpoint
ENDPOINT_RESOURCENAME=projects/992044206639/locations/us-central1/endpoints/1416074769308057600
ENDPOINT_ID=1416074769308057600


Using endpoint [https://us-central1-aiplatform.googleapis.com/]
Waiting for operation [4378990282377527296]...


Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.